# Programación Orientada a Objetos + CRUD con Python
## Listas, diccionarios y persistencia en Excel

Este notebook continúa el aprendizaje de **Programación Orientada a Objetos (POO)** y lo conecta con uno de los patrones más utilizados en aplicaciones reales: **CRUD**.

Trabajaremos con las mismas áreas del notebook anterior:

1. **Salud** → Pacientes
2. **Logística** → Pedidos
3. **Educación** → Estudiantes
4. **Comercio** → Productos

El objetivo es comprender cómo una clase puede utilizar **listas y diccionarios** para administrar múltiples registros y, finalmente, **guardar y recuperar esos datos desde Excel**.

# Objetivos de aprendizaje

Al finalizar podrás:

- Relacionar **POO** con estructuras de datos como listas y diccionarios.
- Comprender el significado de **CRUD**.
- Crear registros.
- Consultar uno o todos los registros.
- Actualizar registros existentes.
- Eliminar registros.
- Buscar registros mediante un identificador.
- Convertir objetos a diccionarios.
- Convertir una lista de diccionarios en un `DataFrame`.
- Exportar información a Excel.
- Cargar nuevamente un Excel.
- Continuar realizando CRUD después de cargar los datos.
- Construir un CRUD persistente que trabaje directamente con un archivo Excel.

# ¿Qué significa CRUD?

CRUD representa las cuatro operaciones básicas sobre datos:

| Letra | Operación | Acción |
|---|---|---|
| **C** | Create | Crear un nuevo registro |
| **R** | Read | Leer o consultar registros |
| **U** | Update | Actualizar un registro |
| **D** | Delete | Eliminar un registro |

Ejemplo aplicado a pacientes:

```text
CREATE → Registrar un paciente
READ   → Consultar pacientes
UPDATE → Actualizar el peso de un paciente
DELETE → Eliminar un paciente
```

# Relación entre POO y CRUD

Usaremos dos tipos de clases:

```text
Clase Entidad
    ↓
Representa un registro individual
Ejemplo: Paciente

Clase Gestora
    ↓
Administra múltiples registros
Ejemplo: GestorPacientes
```

La clase gestora almacenará los datos inicialmente en:

```python
self.registros = []
```

Cada elemento de esa lista será un **diccionario**.

# Preparación

Utilizaremos `pandas` para trabajar con tablas y archivos Excel.

> En Google Colab normalmente `pandas` ya está disponible.

In [ ]:
import pandas as pd


# Parte 1. Salud
## Construcción completa de POO + CRUD desde cero

Comenzaremos desarrollando el proceso con más detalle utilizando un sistema sencillo de **gestión de pacientes**.

## Paso 1. Crear una entidad con POO

Cada paciente será un objeto.

In [ ]:
class Paciente:

    def __init__(self, id_paciente, nombre, edad, peso):
        self.id_paciente = id_paciente
        self.nombre = nombre
        self.edad = edad
        self.peso = peso


In [ ]:
paciente_1 = Paciente(
    id_paciente=1,
    nombre="Ana",
    edad=35,
    peso=68
)

print(paciente_1.nombre)
print(paciente_1.peso)


## Paso 2. Convertir un objeto en diccionario

Para administrar muchos objetos y posteriormente trabajar con tablas, resulta útil convertir cada objeto a un diccionario.

In [ ]:
class Paciente:

    def __init__(self, id_paciente, nombre, edad, peso):
        self.id_paciente = id_paciente
        self.nombre = nombre
        self.edad = edad
        self.peso = peso

    def to_dict(self):
        return {
            "id_paciente": self.id_paciente,
            "nombre": self.nombre,
            "edad": self.edad,
            "peso": self.peso
        }


In [ ]:
paciente_1 = Paciente(1, "Ana", 35, 68)

registro = paciente_1.to_dict()

print(registro)
print(type(registro))


Tenemos ahora:

```text
Objeto Paciente
      ↓
   to_dict()
      ↓
Diccionario
```

Esto permitirá guardar varios registros dentro de una lista.

## Paso 3. Crear una clase gestora

La clase `GestorPacientes` se encargará de administrar todos los registros.

Inicialmente tendrá una lista vacía.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []


In [ ]:
gestor = GestorPacientes()

print(gestor.registros)


# C — CREATE
## Crear registros

El primer método CRUD será `crear()`.

Recibirá un objeto `Paciente` y almacenará su representación como diccionario.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        self.registros.append(
            paciente.to_dict()
        )


In [ ]:
gestor = GestorPacientes()

gestor.crear(
    Paciente(1, "Ana", 35, 68)
)

gestor.crear(
    Paciente(2, "Luis", 52, 82)
)

print(gestor.registros)


Cada ejecución de `crear()` agrega un nuevo diccionario a la lista.

```text
self.registros
│
├── {Paciente 1}
└── {Paciente 2}
```

# R — READ
## Leer todos los registros

Ahora agregaremos un método que retorne la colección completa.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        self.registros.append(paciente.to_dict())

    def leer_todos(self):
        return self.registros


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))
gestor.crear(Paciente(3, "Marta", 41, 71))

for paciente in gestor.leer_todos():
    print(paciente)


## Leer un registro específico

Para consultar un solo paciente utilizaremos su `id_paciente`.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        self.registros.append(paciente.to_dict())

    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):

        for paciente in self.registros:

            if paciente["id_paciente"] == id_paciente:
                return paciente

        return None


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))

resultado = gestor.buscar(2)

print(resultado)


# U — UPDATE
## Actualizar un registro

Utilizaremos el identificador para encontrar al paciente y actualizaremos solamente los valores enviados al método.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        self.registros.append(paciente.to_dict())

    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):
        for paciente in self.registros:
            if paciente["id_paciente"] == id_paciente:
                return paciente
        return None

    def actualizar(self, id_paciente, **cambios):

        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        paciente.update(cambios)

        return True


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))

gestor.actualizar(
    1,
    peso=65,
    edad=36
)

print(gestor.buscar(1))


### ¿Qué hace `**cambios`?

Permite enviar distintos campos al método:

```python
gestor.actualizar(1, peso=65)
```

o:

```python
gestor.actualizar(1, peso=65, edad=36)
```

Dentro del método, Python los recibe como un diccionario.

In [ ]:
def ejemplo_cambios(**cambios):
    print(cambios)


ejemplo_cambios(
    peso=65,
    edad=36
)


# D — DELETE
## Eliminar un registro

Buscaremos el identificador y eliminaremos el diccionario correspondiente.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        self.registros.append(paciente.to_dict())

    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):
        for paciente in self.registros:
            if paciente["id_paciente"] == id_paciente:
                return paciente
        return None

    def actualizar(self, id_paciente, **cambios):
        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        paciente.update(cambios)
        return True

    def eliminar(self, id_paciente):

        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        self.registros.remove(paciente)

        return True


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))
gestor.crear(Paciente(3, "Marta", 41, 71))

gestor.eliminar(2)

print(gestor.leer_todos())


# Clase completa: CRUD de Pacientes

Integramos ahora las cuatro operaciones en una sola clase.

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    # CREATE
    def crear(self, paciente):

        if self.buscar(paciente.id_paciente) is not None:
            print("Ya existe un paciente con ese ID.")
            return False

        self.registros.append(
            paciente.to_dict()
        )

        return True

    # READ
    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):

        for paciente in self.registros:

            if paciente["id_paciente"] == id_paciente:
                return paciente

        return None

    # UPDATE
    def actualizar(self, id_paciente, **cambios):

        paciente = self.buscar(id_paciente)

        if paciente is None:
            print("Paciente no encontrado.")
            return False

        paciente.update(cambios)

        return True

    # DELETE
    def eliminar(self, id_paciente):

        paciente = self.buscar(id_paciente)

        if paciente is None:
            print("Paciente no encontrado.")
            return False

        self.registros.remove(paciente)

        return True


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))
gestor.crear(Paciente(3, "Marta", 41, 71))

print("REGISTROS INICIALES")
print(gestor.leer_todos())

gestor.actualizar(2, peso=79)

gestor.eliminar(3)

print("\nREGISTROS FINALES")
print(gestor.leer_todos())


# De lista de diccionarios a tabla

`pandas` puede convertir directamente nuestra lista de diccionarios en un `DataFrame`.

In [ ]:
df_pacientes = pd.DataFrame(
    gestor.leer_todos()
)

df_pacientes


# Exportar el CRUD a Excel

Ahora agregaremos un método `exportar_excel()`.

El flujo será:

```text
Lista de diccionarios
        ↓
DataFrame
        ↓
Excel
```

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        if self.buscar(paciente.id_paciente) is not None:
            return False

        self.registros.append(paciente.to_dict())
        return True

    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):
        for paciente in self.registros:
            if paciente["id_paciente"] == id_paciente:
                return paciente
        return None

    def actualizar(self, id_paciente, **cambios):
        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        paciente.update(cambios)
        return True

    def eliminar(self, id_paciente):
        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        self.registros.remove(paciente)
        return True

    def exportar_excel(self, archivo="pacientes.xlsx"):

        df = pd.DataFrame(self.registros)

        df.to_excel(
            archivo,
            index=False
        )

        print(f"Datos exportados a: {archivo}")


In [ ]:
gestor = GestorPacientes()

gestor.crear(Paciente(1, "Ana", 35, 68))
gestor.crear(Paciente(2, "Luis", 52, 82))
gestor.crear(Paciente(3, "Marta", 41, 71))

gestor.exportar_excel(
    "pacientes.xlsx"
)


# Cargar nuevamente el Excel

Para que CRUD no termine cuando cerramos Python, agregaremos `cargar_excel()`.

El archivo Excel se convierte nuevamente en:

```text
Excel
  ↓
DataFrame
  ↓
Lista de diccionarios
  ↓
self.registros
```

In [ ]:
class GestorPacientes:

    def __init__(self):
        self.registros = []

    def crear(self, paciente):
        if self.buscar(paciente.id_paciente) is not None:
            return False

        self.registros.append(paciente.to_dict())
        return True

    def leer_todos(self):
        return self.registros

    def buscar(self, id_paciente):
        for paciente in self.registros:
            if paciente["id_paciente"] == id_paciente:
                return paciente
        return None

    def actualizar(self, id_paciente, **cambios):
        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        paciente.update(cambios)
        return True

    def eliminar(self, id_paciente):
        paciente = self.buscar(id_paciente)

        if paciente is None:
            return False

        self.registros.remove(paciente)
        return True

    def exportar_excel(self, archivo="pacientes.xlsx"):
        pd.DataFrame(
            self.registros
        ).to_excel(
            archivo,
            index=False
        )

    def cargar_excel(self, archivo="pacientes.xlsx"):

        df = pd.read_excel(archivo)

        self.registros = df.to_dict(
            orient="records"
        )

        print(
            f"{len(self.registros)} registros cargados."
        )


## Recuperar los datos y continuar el CRUD

Crearemos un gestor nuevo, cargaremos el Excel y después aplicaremos UPDATE, DELETE y CREATE.

In [ ]:
gestor_recuperado = GestorPacientes()

gestor_recuperado.cargar_excel(
    "pacientes.xlsx"
)

print(
    gestor_recuperado.leer_todos()
)


In [ ]:
# UPDATE después de cargar Excel
gestor_recuperado.actualizar(
    1,
    peso=64
)

# DELETE después de cargar Excel
gestor_recuperado.eliminar(2)

# CREATE después de cargar Excel
gestor_recuperado.crear(
    Paciente(4, "Carlos", 44, 76)
)

pd.DataFrame(
    gestor_recuperado.leer_todos()
)


Los cambios todavía están en memoria.  
Para persistirlos debemos volver a guardar el Excel.

In [ ]:
gestor_recuperado.exportar_excel(
    "pacientes_actualizados.xlsx"
)


# Nivel 2. CRUD persistente sobre Excel

Hasta ahora:

```text
CRUD → memoria → exportación manual
```

Ahora construiremos una clase donde cada operación:

1. carga el Excel,
2. realiza el CRUD,
3. guarda nuevamente el Excel.

Conceptualmente:

```text
CREATE ─┐
READ   ─┤
UPDATE ─┼──→ archivo Excel
DELETE ─┘
```

Esto permite usar el archivo `.xlsx` como una fuente de datos sencilla para fines educativos.

In [ ]:
class CRUDPacientesExcel:

    def __init__(self, archivo="crud_pacientes.xlsx"):
        self.archivo = archivo

    def _cargar(self):

        try:
            return pd.read_excel(
                self.archivo
            )

        except FileNotFoundError:

            return pd.DataFrame(
                columns=[
                    "id_paciente",
                    "nombre",
                    "edad",
                    "peso"
                ]
            )

    def _guardar(self, df):

        df.to_excel(
            self.archivo,
            index=False
        )

    # CREATE
    def crear(self, paciente):

        df = self._cargar()

        if paciente.id_paciente in df["id_paciente"].values:
            print("El ID ya existe.")
            return False

        nuevo = pd.DataFrame([
            paciente.to_dict()
        ])

        df = pd.concat(
            [df, nuevo],
            ignore_index=True
        )

        self._guardar(df)

        return True

    # READ
    def leer_todos(self):

        return self._cargar()

    def buscar(self, id_paciente):

        df = self._cargar()

        resultado = df[
            df["id_paciente"] == id_paciente
        ]

        return resultado

    # UPDATE
    def actualizar(self, id_paciente, **cambios):

        df = self._cargar()

        condicion = (
            df["id_paciente"] == id_paciente
        )

        if not condicion.any():
            print("Paciente no encontrado.")
            return False

        for campo, valor in cambios.items():

            if campo in df.columns:
                df.loc[
                    condicion,
                    campo
                ] = valor

        self._guardar(df)

        return True

    # DELETE
    def eliminar(self, id_paciente):

        df = self._cargar()

        condicion = (
            df["id_paciente"] == id_paciente
        )

        if not condicion.any():
            print("Paciente no encontrado.")
            return False

        df = df[
            ~condicion
        ]

        self._guardar(df)

        return True


## CREATE directamente sobre Excel

In [ ]:
crud_excel = CRUDPacientesExcel()

crud_excel.crear(
    Paciente(101, "Andrea", 29, 61)
)

crud_excel.crear(
    Paciente(102, "Miguel", 47, 80)
)

crud_excel.crear(
    Paciente(103, "Sofía", 38, 67)
)


## READ directamente desde Excel

In [ ]:
crud_excel.leer_todos()


## UPDATE directamente sobre Excel

In [ ]:
crud_excel.actualizar(
    102,
    peso=77,
    edad=48
)

crud_excel.leer_todos()


## DELETE directamente sobre Excel

In [ ]:
crud_excel.eliminar(103)

crud_excel.leer_todos()


# Parte 2. Logística
## POO + CRUD de Pedidos

Aplicaremos ahora exactamente el mismo patrón a otra área.

In [ ]:
class Pedido:

    def __init__(
        self,
        id_pedido,
        cliente,
        peso,
        estado
    ):
        self.id_pedido = id_pedido
        self.cliente = cliente
        self.peso = peso
        self.estado = estado

    def to_dict(self):
        return {
            "id_pedido": self.id_pedido,
            "cliente": self.cliente,
            "peso": self.peso,
            "estado": self.estado
        }


In [ ]:
class GestorPedidos:

    def __init__(self):
        self.registros = []

    # CREATE
    def crear(self, pedido):

        if self.buscar(pedido.id_pedido):
            return False

        self.registros.append(
            pedido.to_dict()
        )

        return True

    # READ
    def leer_todos(self):
        return self.registros

    def buscar(self, id_pedido):

        for pedido in self.registros:

            if pedido["id_pedido"] == id_pedido:
                return pedido

        return None

    # UPDATE
    def actualizar(self, id_pedido, **cambios):

        pedido = self.buscar(id_pedido)

        if pedido is None:
            return False

        pedido.update(cambios)

        return True

    # DELETE
    def eliminar(self, id_pedido):

        pedido = self.buscar(id_pedido)

        if pedido is None:
            return False

        self.registros.remove(pedido)

        return True

    # EXCEL
    def exportar_excel(
        self,
        archivo="pedidos.xlsx"
    ):
        pd.DataFrame(
            self.registros
        ).to_excel(
            archivo,
            index=False
        )

    def cargar_excel(
        self,
        archivo="pedidos.xlsx"
    ):
        df = pd.read_excel(archivo)

        self.registros = df.to_dict(
            orient="records"
        )


In [ ]:
pedidos = GestorPedidos()

pedidos.crear(
    Pedido(
        1,
        "Empresa ABC",
        12.5,
        "Pendiente"
    )
)

pedidos.crear(
    Pedido(
        2,
        "Empresa XYZ",
        8.2,
        "Despachado"
    )
)

pedidos.crear(
    Pedido(
        3,
        "Comercial Norte",
        18.4,
        "Pendiente"
    )
)

pd.DataFrame(
    pedidos.leer_todos()
)


## UPDATE y DELETE de pedidos

In [ ]:
pedidos.actualizar(
    1,
    estado="Despachado"
)

pedidos.eliminar(2)

pd.DataFrame(
    pedidos.leer_todos()
)


## Exportar pedidos a Excel

In [ ]:
pedidos.exportar_excel(
    "pedidos.xlsx"
)


## Recuperar pedidos desde Excel

In [ ]:
pedidos_recuperados = GestorPedidos()

pedidos_recuperados.cargar_excel(
    "pedidos.xlsx"
)

pd.DataFrame(
    pedidos_recuperados.leer_todos()
)


# Parte 3. Educación
## POO + CRUD de Estudiantes

La entidad será ahora `Estudiante`.

In [ ]:
class Estudiante:

    def __init__(
        self,
        id_estudiante,
        nombre,
        carrera,
        nota
    ):
        self.id_estudiante = id_estudiante
        self.nombre = nombre
        self.carrera = carrera
        self.nota = nota

    def to_dict(self):
        return {
            "id_estudiante": self.id_estudiante,
            "nombre": self.nombre,
            "carrera": self.carrera,
            "nota": self.nota
        }


In [ ]:
class GestorEstudiantes:

    def __init__(self):
        self.registros = []

    def crear(self, estudiante):

        if self.buscar(
            estudiante.id_estudiante
        ):
            return False

        self.registros.append(
            estudiante.to_dict()
        )

        return True

    def leer_todos(self):
        return self.registros

    def buscar(self, id_estudiante):

        for estudiante in self.registros:

            if (
                estudiante["id_estudiante"]
                == id_estudiante
            ):
                return estudiante

        return None

    def actualizar(
        self,
        id_estudiante,
        **cambios
    ):
        estudiante = self.buscar(
            id_estudiante
        )

        if estudiante is None:
            return False

        estudiante.update(cambios)

        return True

    def eliminar(self, id_estudiante):

        estudiante = self.buscar(
            id_estudiante
        )

        if estudiante is None:
            return False

        self.registros.remove(
            estudiante
        )

        return True

    def exportar_excel(
        self,
        archivo="estudiantes.xlsx"
    ):
        pd.DataFrame(
            self.registros
        ).to_excel(
            archivo,
            index=False
        )

    def cargar_excel(
        self,
        archivo="estudiantes.xlsx"
    ):
        df = pd.read_excel(archivo)

        self.registros = df.to_dict(
            orient="records"
        )


In [ ]:
estudiantes = GestorEstudiantes()

estudiantes.crear(
    Estudiante(
        1,
        "María",
        "Ingeniería",
        8.7
    )
)

estudiantes.crear(
    Estudiante(
        2,
        "José",
        "Administración",
        7.5
    )
)

estudiantes.crear(
    Estudiante(
        3,
        "Daniela",
        "Economía",
        9.1
    )
)

pd.DataFrame(
    estudiantes.leer_todos()
)


## Actualizar una calificación

In [ ]:
estudiantes.actualizar(
    2,
    nota=8.2
)

pd.DataFrame(
    estudiantes.leer_todos()
)


## Eliminar un estudiante

In [ ]:
estudiantes.eliminar(1)

pd.DataFrame(
    estudiantes.leer_todos()
)


## Exportar y recuperar desde Excel

In [ ]:
estudiantes.exportar_excel(
    "estudiantes.xlsx"
)

estudiantes_excel = GestorEstudiantes()

estudiantes_excel.cargar_excel(
    "estudiantes.xlsx"
)

pd.DataFrame(
    estudiantes_excel.leer_todos()
)


# Parte 4. Comercio
## POO + CRUD de Productos

Finalmente utilizaremos una entidad `Producto`.

In [ ]:
class Producto:

    def __init__(
        self,
        id_producto,
        nombre,
        precio,
        stock
    ):
        self.id_producto = id_producto
        self.nombre = nombre
        self.precio = precio
        self.stock = stock

    def to_dict(self):
        return {
            "id_producto": self.id_producto,
            "nombre": self.nombre,
            "precio": self.precio,
            "stock": self.stock
        }


In [ ]:
class GestorProductos:

    def __init__(self):
        self.registros = []

    def crear(self, producto):

        if self.buscar(
            producto.id_producto
        ):
            return False

        self.registros.append(
            producto.to_dict()
        )

        return True

    def leer_todos(self):
        return self.registros

    def buscar(self, id_producto):

        for producto in self.registros:

            if (
                producto["id_producto"]
                == id_producto
            ):
                return producto

        return None

    def actualizar(
        self,
        id_producto,
        **cambios
    ):
        producto = self.buscar(
            id_producto
        )

        if producto is None:
            return False

        producto.update(cambios)

        return True

    def eliminar(self, id_producto):

        producto = self.buscar(
            id_producto
        )

        if producto is None:
            return False

        self.registros.remove(
            producto
        )

        return True

    def exportar_excel(
        self,
        archivo="productos.xlsx"
    ):
        pd.DataFrame(
            self.registros
        ).to_excel(
            archivo,
            index=False
        )

    def cargar_excel(
        self,
        archivo="productos.xlsx"
    ):
        df = pd.read_excel(archivo)

        self.registros = df.to_dict(
            orient="records"
        )


In [ ]:
productos = GestorProductos()

productos.crear(
    Producto(
        1,
        "Laptop",
        950,
        10
    )
)

productos.crear(
    Producto(
        2,
        "Monitor",
        280,
        15
    )
)

productos.crear(
    Producto(
        3,
        "Teclado",
        45,
        30
    )
)

pd.DataFrame(
    productos.leer_todos()
)


## UPDATE: cambiar precio y stock

In [ ]:
productos.actualizar(
    2,
    precio=265,
    stock=18
)

pd.DataFrame(
    productos.leer_todos()
)


## DELETE: eliminar un producto

In [ ]:
productos.eliminar(3)

pd.DataFrame(
    productos.leer_todos()
)


## Exportar y recuperar productos

In [ ]:
productos.exportar_excel(
    "productos.xlsx"
)

productos_recuperados = GestorProductos()

productos_recuperados.cargar_excel(
    "productos.xlsx"
)

pd.DataFrame(
    productos_recuperados.leer_todos()
)


# Parte 5. Detectar el patrón repetido

Observa que en todas las áreas estamos repitiendo prácticamente la misma estructura:

```text
Entidad
  ↓
to_dict()
  ↓
Gestor
  ↓
CREATE
READ
UPDATE
DELETE
  ↓
Excel
```

Esto nos permite introducir una idea importante de POO:

> Si varias clases comparten el mismo comportamiento, podemos reutilizar código mediante una clase general.

# CRUD genérico reutilizable

Construiremos un gestor capaz de trabajar con diferentes entidades.

Recibirá:

- nombre del campo identificador
- nombre del archivo Excel

In [ ]:
class GestorCRUD:

    def __init__(
        self,
        campo_id,
        archivo
    ):
        self.campo_id = campo_id
        self.archivo = archivo
        self.registros = []

    def crear(self, objeto):

        registro = objeto.to_dict()

        valor_id = registro[
            self.campo_id
        ]

        if self.buscar(valor_id):
            return False

        self.registros.append(
            registro
        )

        return True

    def leer_todos(self):
        return self.registros

    def buscar(self, valor_id):

        for registro in self.registros:

            if (
                registro[self.campo_id]
                == valor_id
            ):
                return registro

        return None

    def actualizar(
        self,
        valor_id,
        **cambios
    ):
        registro = self.buscar(
            valor_id
        )

        if registro is None:
            return False

        registro.update(cambios)

        return True

    def eliminar(self, valor_id):

        registro = self.buscar(
            valor_id
        )

        if registro is None:
            return False

        self.registros.remove(
            registro
        )

        return True

    def exportar_excel(self):

        pd.DataFrame(
            self.registros
        ).to_excel(
            self.archivo,
            index=False
        )

    def cargar_excel(self):

        df = pd.read_excel(
            self.archivo
        )

        self.registros = df.to_dict(
            orient="records"
        )


## El mismo CRUD para pacientes

In [ ]:
crud_pacientes = GestorCRUD(
    campo_id="id_paciente",
    archivo="crud_generico_pacientes.xlsx"
)

crud_pacientes.crear(
    Paciente(1, "Ana", 35, 68)
)

crud_pacientes.crear(
    Paciente(2, "Luis", 52, 82)
)

pd.DataFrame(
    crud_pacientes.leer_todos()
)


## El mismo CRUD para productos

In [ ]:
crud_productos = GestorCRUD(
    campo_id="id_producto",
    archivo="crud_generico_productos.xlsx"
)

crud_productos.crear(
    Producto(1, "Laptop", 950, 10)
)

crud_productos.crear(
    Producto(2, "Monitor", 280, 15)
)

crud_productos.actualizar(
    1,
    stock=8
)

pd.DataFrame(
    crud_productos.leer_todos()
)


# Parte 6. Flujo completo de una aplicación sencilla

El proceso completo que hemos construido es:

```text
1. Crear clase de la entidad
           ↓
2. Crear objetos
           ↓
3. Convertir objetos a diccionarios
           ↓
4. Guardarlos en una lista
           ↓
5. Aplicar CRUD
           ↓
6. Convertir lista a DataFrame
           ↓
7. Exportar a Excel
           ↓
8. Cargar Excel
           ↓
9. Recuperar lista de diccionarios
           ↓
10. Continuar CRUD
           ↓
11. Guardar nuevamente
```

# Comparación entre las cuatro áreas

| Área | Entidad | Identificador | Ejemplo UPDATE | Archivo |
|---|---|---|---|---|
| Salud | `Paciente` | `id_paciente` | cambiar peso | pacientes.xlsx |
| Logística | `Pedido` | `id_pedido` | cambiar estado | pedidos.xlsx |
| Educación | `Estudiante` | `id_estudiante` | cambiar nota | estudiantes.xlsx |
| Comercio | `Producto` | `id_producto` | cambiar precio/stock | productos.xlsx |

# ¿Dónde está POO y dónde está CRUD?

### POO

```python
class Paciente:
    ...
```

Representa el modelo del dato.

### CRUD

```python
crear()
buscar()
actualizar()
eliminar()
```

Representa las operaciones sobre los datos.

### Lista y diccionarios

```python
self.registros = []
```

Representan el almacenamiento temporal.

### Excel

```python
df.to_excel(...)
pd.read_excel(...)
```

Representa la persistencia.

# Ejercicio 1. CREATE y READ

Utilizando el ejemplo de productos:

1. Crea tres productos nuevos.
2. Muéstralos como `DataFrame`.
3. Busca un producto específico por ID.

In [ ]:
# Escribe aquí tu solución


# Ejercicio 2. UPDATE y DELETE

Utilizando estudiantes:

1. Actualiza la nota de un estudiante.
2. Actualiza también su carrera.
3. Elimina otro estudiante.
4. Muestra el resultado final.

In [ ]:
# Escribe aquí tu solución


# Ejercicio 3. Persistencia

Selecciona cualquiera de las cuatro áreas:

1. Crea al menos cinco registros.
2. Exporta los registros a Excel.
3. Crea un gestor nuevo.
4. Carga el archivo Excel.
5. Actualiza un registro.
6. Elimina otro registro.
7. Agrega un registro nuevo.
8. Guarda el resultado en otro Excel.

In [ ]:
# Escribe aquí tu solución


# Ejercicio 4. Nueva área

Crea un CRUD POO para una de estas áreas:

- Banco → `Cuenta`
- Recursos Humanos → `Empleado`
- Transporte → `Vehiculo`
- Turismo → `Reserva`
- Manufactura → `Maquina`

La entidad debe incluir:

- identificador único
- mínimo tres atributos adicionales
- método `to_dict()`

El gestor debe incluir:

- `crear()`
- `leer_todos()`
- `buscar()`
- `actualizar()`
- `eliminar()`
- `exportar_excel()`
- `cargar_excel()`

In [ ]:
# RETO FINAL


# Conclusión

En este notebook conectamos tres ideas fundamentales:

```text
POO
 ↓
CRUD
 ↓
Persistencia
```

La **POO** permite representar las entidades del problema.

El **CRUD** permite administrar sus registros.

Las **listas y diccionarios** permiten almacenar los datos temporalmente.

Un **DataFrame** permite transformar esos registros en una estructura tabular.

Finalmente, **Excel** permite conservar los datos y recuperarlos posteriormente.

Este patrón constituye una excelente transición antes de trabajar con:

- archivos CSV,
- bases de datos SQL,
- SQLite,
- APIs,
- aplicaciones Streamlit.